# 05 · Deep models on CPU

EEGNet, ShallowConvNet, DeepConvNet, a tiny LSTM and a tiny Transformer — all via braindecode/skorch, all trainable on CPU in minutes thanks to small inputs and few epochs.

Per-channel scaling is fit on train only (do it inside a pipeline or on the train split).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt


In [ ]:
from eeglog.data import load_moabb
from eeglog.models_deep import build_eegnet, as_float32
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
d = load_moabb(subjects=[1], paradigm='left_right')
X = as_float32(d.X); y = (d.y == d.y[0]).astype('int64')  # encode to 0/1
n_ch, n_t = X.shape[1], X.shape[2]

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, stratify=y, random_state=42)
clf = build_eegnet(n_ch, n_t, n_classes=2, max_epochs=30)
clf.fit(Xtr, ytr)
print('balanced acc:', balanced_accuracy_score(yte, clf.predict(Xte)))

In [ ]:
# Training/valid loss curve from skorch history.
h = clf.history
plt.plot(h[:, 'train_loss'], label='train')
plt.plot(h[:, 'valid_loss'], label='valid')
plt.legend(); plt.xlabel('epoch'); plt.title('EEGNet learning curve')

**Checkpoint:** a deep net trained on CPU with a sane learning curve. Swap in `build_shallow / build_deep` to compare CNNs. For the recurrent/attention nets, decimate the time axis first so they stay fast on CPU:

```python
from eeglog.models_deep import build_transformer, decimate_time
Xd = decimate_time(d.X, factor=4)   # 1001 -> 251 samples
clf = build_transformer(Xd.shape[1], Xd.shape[2], 2)
```

The honest cross-subject comparison is in notebook 06.